<a href="https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1019/semantic_distance_layer_probe_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1019 v2 — Layer-wise changes in suppressed-concept representations

**Main question:** How does the semantic distance of a cue affect X-vs-Y decodability **before versus after reading the cue**? Does that relationship differ when concept X is explicitly suppressed?

**Design:** 8 fixed concept pairs × 12 prespecified cue sentences each × 2 instruction conditions (`neutral` and `suppression`) × all 24 transformer layers in Qwen2.5-0.5B-Instruct. A second, separate sentence encoder estimates semantic distance. Probe training uses multiple named and name-free concept descriptions; held-out descriptions are distinct from training examples. Each experimental input contains two explicit `STATE:` measurement markers. Readouts use the **same model forward pass**, leveraging the causal mask so the pre-cue marker cannot attend to the cue.

**Caveats:** These are *token-position differences in an autoregressive prompt*, not evidence of thinking, forgetting, temporal memory loss, or a causal mechanism. X remains written once in both instructions. Probe scores represent **X-vs-Y decodability** rather than absolute concept strength. The experiment is exploratory; 8 concept pairs are not representative of all language models or categories.

**Output location:** `GraduationThesis/1019/results_v2/<UTC-run-id>/` in Colab; optionally upload CSV/PNG/JSON to the same path in GitHub using Colab Secret `GITHUB_RESULTS_TOKEN`. A downloadable ZIP is created regardless of token availability.


## 0. Colab setup

Use **Runtime → Change runtime type → T4 GPU**. Avoid replacing Colab's PyTorch/CUDA build. Audio and vision packages are not required for text-only modeling.

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install "transformers>=4.54,<5" "scikit-learn>=1.3" "scipy>=1.11" "pandas>=2.1" "matplotlib>=3.8" "tqdm>=4.65" "requests>=2.31"

In [ ]:
import os, io, re, json, sys, random, zipfile, base64, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import requests
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

SEED=42
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
EMBED_ID="sentence-transformers/all-MiniLM-L6-v2"
MODEL_LAYERS=list(range(1,25))       # every transformer layer: 1..24; 0 is token embedding
MAIN_LAYER=24                       # prespecified confirmatory-focus layer
AUC_CUTOFF=0.75                    # exploratory probe reliability screen
MIN_VALIDATED_FOR_LAYER_PLOT=5
CONDITIONS=["neutral","suppression"]
PROPOSED_LEVELS=["near","medium","far","unrelated"]
N_BOOT=2000
UPLOAD_TO_GITHUB=True
REPO="ryouy/GraduationThesis"
BRANCH="main"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
if DEVICE=="cpu":warnings.warn("No CUDA: select Colab T4 GPU for timely execution.")
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT=Path("/content/GraduationThesis") if Path("/content").is_dir() else Path.cwd()/"GraduationThesis"
OUT=ROOT/"1019"/"results_v2"/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
def save_csv(frame,name):
    frame.to_csv(OUT/name,index=False)
    print(f"Saved {name}: {len(frame)} rows")
def save_figure(name):
    plt.tight_layout();plt.savefig(OUT/name,dpi=190,bbox_inches="tight");plt.show();plt.close()
print("Device:",DEVICE,"| output:",OUT)

# Pin both auditable input banks to exact repository commits.
STIMULI_URL=("https://raw.githubusercontent.com/ryouy/GraduationThesis/"
 "88579fa455f8f3f75085d395bcf30a313ba400dc/1019/stimuli_v2.csv")
DESCRIPTION_URL=("https://raw.githubusercontent.com/ryouy/GraduationThesis/"
 "a6e058ed05c5a43e28cf46bdecd10eb54edcf19d/1019/probe_description_bank_v2.json")
def remote_bytes(url):
    r=requests.get(url,timeout=60);r.raise_for_status()
    return r.content
stimuli_bytes=remote_bytes(STIMULI_URL)
descriptions_bytes=remote_bytes(DESCRIPTION_URL)
cues=pd.read_csv(io.BytesIO(stimuli_bytes))
desc=json.loads(descriptions_bytes.decode("utf-8"))
CONCEPTS=sorted(cues.concept.unique().tolist())
assert len(CONCEPTS)==8 and len(cues)==96
assert set(cues.proposed_level)==set(PROPOSED_LEVELS)
assert cues.groupby(["concept","proposed_level"]).size().eq(3).all()
assert set(CONCEPTS)==set(desc)
def contains_word(s,w):
    return bool(re.search(r"(?<![A-Za-z])"+re.escape(w)+r"(?![A-Za-z])",s,re.I))
for row in cues.itertuples():
    assert not contains_word(row.cue_text,row.concept)
    assert not contains_word(row.cue_text,row.replacement)
for x,d in desc.items():
    for k in ["x_train","x_test","y_train","y_test"]:
        assert len(d[k])==3 and all(not contains_word(v,x) and
            not contains_word(v,cues.loc[cues.concept==x,"replacement"].iloc[0]) for v in d[k])
# Save the exact bank of prewritten inputs with every run.
(OUT/"stimuli_v2.csv").write_bytes(stimuli_bytes)
(OUT/"probe_description_bank_v2.json").write_bytes(descriptions_bytes)
INPUT_HASHES={"stimuli_sha256":hashlib.sha256(stimuli_bytes).hexdigest(),
              "descriptions_sha256":hashlib.sha256(descriptions_bytes).hexdigest()}
print("Input bank validated:",len(cues),"cues, 8 pairs. SHA256:",INPUT_HASHES)
display(cues.groupby(["concept","proposed_level"]).size().unstack())

## 1. Independent meaning-distance measurement

MiniLM estimates 1 − cosine between the cue and two short target-name sentences. Target distances are not evaluated using Qwen's hidden states or the learned X/Y probes. **Prewritten labels are audited, not assumed correct.** For descriptive quartile plots, sort the 12 cues per concept by measured distance into four equal groups of three; the continuous measured distance is the actual predictor. These are *embedding-based*, not human-validated distance categories.

In [ ]:
e_tok=AutoTokenizer.from_pretrained(EMBED_ID,trust_remote_code=False)
e_model=AutoModel.from_pretrained(EMBED_ID,trust_remote_code=False).cpu().eval()
def embed_sentences(texts,batch_size=32):
    result=[]
    for start in tqdm(range(0,len(texts),batch_size),desc="Independent semantic embeddings"):
        b=e_tok(texts[start:start+batch_size],padding=True,truncation=True,
                max_length=128,return_tensors="pt")
        with torch.inference_mode():
            h=e_model(**b).last_hidden_state
            mask=b.attention_mask.unsqueeze(-1)
            pooled=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
            result.append(F.normalize(pooled.float(),dim=-1).cpu().numpy())
    return np.concatenate(result,axis=0)
anchors=[f"This text is about {x}." for x in CONCEPTS]+[
         f"The main topic is {x}." for x in CONCEPTS]
embedding=embed_sentences(anchors+cues.cue_text.tolist())
centers={}
for i,x in enumerate(CONCEPTS):
    center=(embedding[i]+embedding[i+len(CONCEPTS)])/2
    centers[x]=center/np.linalg.norm(center)
cues["semantic_similarity"]=[float(centers[r.concept]@embedding[2*len(CONCEPTS)+i])
                             for i,r in cues.reset_index(drop=True).iterrows()]
cues["semantic_distance"]=1-cues.semantic_similarity
# Within-concept empirical quartiles, made without seeing Qwen or probe outcomes.
cues=cues.sort_values(["concept","semantic_distance","cue_id"]).reset_index(drop=True)
cues["distance_rank"]=cues.groupby("concept").cumcount()+1
cues["measured_bin"]=cues.distance_rank.map(lambda rank:PROPOSED_LEVELS[(rank-1)//3])
cues["word_count"]=cues.cue_text.str.split().str.len()
audit=cues.groupby(["concept","proposed_level"],as_index=False).semantic_distance.mean()
audit_wide=audit.pivot(index="concept",columns="proposed_level",values="semantic_distance").reindex(columns=PROPOSED_LEVELS)
audit_wide["proposed_order_confirmed"]=np.all(np.diff(audit_wide[PROPOSED_LEVELS].to_numpy(),axis=1)>0,axis=1)
save_csv(cues,"01_stimulus_distances_and_ranks.csv")
save_csv(audit_wide.reset_index(),"02_proposed_distance_audit.csv")
display(audit_wide.round(3))
print("Proposed four-group order holds for",int(audit_wide.proposed_order_confirmed.sum()),"/8 concepts.")
print("Measured bins (quartiles) are descriptive and may reassign cues from their original categories.")
del e_model,e_tok,embedding
if torch.cuda.is_available():torch.cuda.empty_cache()

## 2. Model loading and precise readout token positions

Use `STATE:` as a common measurement anchor in the **user** portion of the Qwen chat message. In a single forward pass, measure the last token of the first `STATE:` **before the cue** and the last token of the second `STATE:` **after the cue**. Fast-tokenizer offset mappings ensure the exact character location matches an actual token; the two marker token IDs are required to match. Unlike v1, these are not assistant-header terminal readouts.

**Causal interpretation:** The pre-cue marker cannot attend to later tokens; it is therefore insensitive to the subsequent cue except for minor numerical/kernel variability. Changes reflect later token representations under added context, not physical passage of time.

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False,use_fast=True)
assert tok.is_fast,"Precise offset-mapped measurement requires a fast tokenizer."
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,dtype=DTYPE,
    low_cpu_mem_usage=True,trust_remote_code=False).to(DEVICE).eval()
assert model.config.num_hidden_layers==24
MARKER="STATE:"
def user_render(text):
    return tok.apply_chat_template([{"role":"user","content":text}],
        tokenize=False,add_generation_prompt=False)
@torch.inference_mode()
def state_readouts(user_text,expected_markers=1):
    rendered=user_render(user_text)
    occurrences=[m.span() for m in re.finditer(re.escape(MARKER),rendered)]
    if len(occurrences)!=expected_markers:
        raise ValueError(f"Expected {expected_markers} marker(s); saw {len(occurrences)}.")
    batch=tok(rendered,return_offsets_mapping=True,add_special_tokens=False,
        return_tensors="pt")
    offsets=batch.pop("offset_mapping")[0].tolist()
    ids=batch["input_ids"][0].tolist()
    positions=[]
    for start,end in occurrences:
        hits=[i for i,(lo,hi) in enumerate(offsets) if lo<=end-1<hi]
        if len(hits)!=1:raise ValueError(f"Cannot locate terminal marker token: {start,end,hits}")
        positions.append(hits[0])
    if len(set(ids[i] for i in positions))!=1:
        raise ValueError("Marker token IDs differ between readouts; investigate tokenizer boundaries.")
    inp={k:v.to(DEVICE) for k,v in batch.items()}
    result=model(**inp,output_hidden_states=True,use_cache=False,return_dict=True)
    states=[]
    for token_pos in positions:
        states.append(np.stack([
            result.hidden_states[l][0,token_pos,:].float().cpu().numpy()
            for l in MODEL_LAYERS
        ],axis=0))
    # shape each (24,hidden_width), order 1..24; no hidden states saved on GitHub by default.
    return states,positions,len(ids)
demo,position,n_tokens=state_readouts(
    "You may think about banana. Continue reading.\nSTATE:\n"
    "A curved yellow fruit was peeled.\nSTATE:\n",expected_markers=2)
assert len(demo)==2 and all(np.isfinite(z).all() for z in demo)
assert position[0]<position[1]
print("Readout positions:",position,"of",n_tokens,"tokens; vectors:",demo[0].shape)
print("Each row corresponds to a transformer layer from 1 to 24.")

## 3. Probe training — semantic descriptions included, held-out descriptions disjoint

Unlike v1, train each layer's probe on **both named and name-free semantic descriptions**. Distinct descriptions and unseen sentence frames make up the held-out name-free evaluation. Each target X is paired with a related alternative Y (e.g., elephant vs giraffe). That constrains the interpretation to *relative* X-vs-Y information. ROC-AUC is reported separately at **all 24 layers**. The same marker and readout are used for train, validation, and main experiments. AUC≥0.75 is a preselected **exploratory reliability gate**, not a guarantee of a valid semantic representation.

In [ ]:
TRAIN_NAME_FRAMES=[
    "The concept is {}.\nSTATE:\n",
    "Focus attention on {}.\nSTATE:\n",
]
TRAIN_DESCRIPTION_FRAMES=[
    "A short definition is {}.\nSTATE:\n",
    "The subject is described as {}.\nSTATE:\n",
]
TEST_DESCRIPTION_FRAMES=[
    "A curator wrote about {}.\nSTATE:\n",
    "An observer reported {}.\nSTATE:\n",
]
TEST_NAME_FRAMES=["The discussion concerns {}.\nSTATE:\n"]

probe_examples=[]
for x in tqdm(CONCEPTS,desc="Construct independent concept probes"):
    y=cues.loc[cues.concept==x,"replacement"].iloc[0]
    bank=desc[x]
    for label,word,prefix in [(1,x,"x"),(0,y,"y")]:
        for frame in TRAIN_NAME_FRAMES:
            prompt=frame.format(word)
            h,_,_=state_readouts(prompt,1)
            probe_examples.append(dict(concept=x,split="train",y=label,prompt=prompt,feature=h[0]))
        for phrase in bank[f"{prefix}_train"]:
            for frame in TRAIN_DESCRIPTION_FRAMES:
                prompt=frame.format(phrase)
                h,_,_=state_readouts(prompt,1)
                probe_examples.append(dict(concept=x,split="train",y=label,prompt=prompt,feature=h[0]))
        for phrase in bank[f"{prefix}_test"]:
            for frame in TEST_DESCRIPTION_FRAMES:
                prompt=frame.format(phrase)
                h,_,_=state_readouts(prompt,1)
                probe_examples.append(dict(concept=x,split="holdout_description",y=label,
                                           prompt=prompt,feature=h[0]))
        for frame in TEST_NAME_FRAMES:
            prompt=frame.format(word)
            h,_,_=state_readouts(prompt,1)
            probe_examples.append(dict(concept=x,split="holdout_named",y=label,
                                       prompt=prompt,feature=h[0]))
probe_data=pd.DataFrame(probe_examples)
assert probe_data.groupby(["concept","split"]).size().unstack().loc[
    :,["train","holdout_description","holdout_named"]].eq([16,12,2]).all().all()
probes={};metrics=[]
for x in CONCEPTS:
    train=probe_data[(probe_data.concept==x)&(probe_data.split=="train")]
    probes[x]={}
    for layer in MODEL_LAYERS:
        ix=layer-1
        X=np.stack([v[ix] for v in train.feature])
        y=train.y.to_numpy()
        clf=make_pipeline(StandardScaler(),
            LogisticRegression(C=0.1,solver="liblinear",random_state=SEED,max_iter=2500))
        clf.fit(X,y)
        decision_sd=max(float(np.std(clf.decision_function(X),ddof=1)),1e-8)
        probes[x][layer]=(clf,decision_sd)
        for split in ["holdout_description","holdout_named"]:
            ev=probe_data[(probe_data.concept==x)&(probe_data.split==split)]
            held=np.stack([v[ix] for v in ev.feature])
            raw=clf.decision_function(held)
            gt=ev.y.to_numpy()
            metrics.append(dict(concept=x,layer=layer,split=split,n=len(gt),
                auc=float(roc_auc_score(gt,raw)),
                accuracy=float(accuracy_score(gt,raw>=0)),
                train_decision_sd=decision_sd))
all_validation=pd.DataFrame(metrics)
validation=all_validation[all_validation.split=="holdout_description"].copy()
validation["valid_probe"]=validation.auc>=AUC_CUTOFF
save_csv(all_validation,"03_probe_validation_all_layers.csv")
save_csv(validation,"04_probe_validation_flags.csv")
counts=validation.groupby("layer",as_index=False).agg(
    n_passed=("valid_probe","sum"),mean_auc=("auc","mean"),
    mean_accuracy=("accuracy","mean"))
save_csv(counts,"05_probe_validation_per_layer.csv")
display(counts.round(3))
print("Final layer valid:",counts.loc[counts.layer==MAIN_LAYER,"n_passed"].iloc[0],"/8")
del probe_data,probe_examples
if torch.cuda.is_available():torch.cuda.empty_cache()

## 4. Primary experiment — two readouts inside **each single sequence**, all 24 layers

Two conditions: `neutral` = permission (`You may think about X`), `suppression` = prohibition (`Do not think about X`). Unlike v1, **both mention X exactly once** and share the remaining sentence. Cue text is identical between the two conditions, and no Y appears in either instruction. The conditions are not perfectly token-length matched, and permission is not psychologically equivalent to a truly neutral instruction; we record those limitations. The same cue is also assessed *before/after* the marker in each sequence.

Besides X/Y probe scores, record a **probe-free hidden-state distance** (`1 − cosine(pre,post)`) and relative L2 displacement. Those measures do not identify a particular concept and must not be interpreted as semantic activation by themselves.

In [ ]:
def condition_instruction(x,condition):
    return {
        "neutral":f"You may think about {x}. Continue reading.",
        "suppression":f"Do not think about {x}. Continue reading."
    }[condition]
def measurement_input(x,condition,cue):
    return condition_instruction(x,condition)+"\nSTATE:\n"+cue+"\nSTATE:\n"
assert contains_word(condition_instruction("banana","neutral"),"banana")
assert contains_word(condition_instruction("banana","suppression"),"banana")
assert all(len(re.findall(r"\b"+re.escape("banana")+r"\b",
           condition_instruction("banana",z)))==1 for z in CONDITIONS)
probe_map={(r.concept,int(r.layer)):(float(r.auc),bool(r.valid_probe))
           for r in validation.itertuples()}
measurements=[];precue_maxdiff={}
for x in tqdm(CONCEPTS,desc="Layer-wise internal state experiment"):
    sub=cues[cues.concept==x]
    for condition in CONDITIONS:
        first_pre=None
        max_error=0.0
        for cue in sub.itertuples():
            p=measurement_input(x,condition,cue.cue_text)
            (pre,post),positions,n_tokens=state_readouts(p,2)
            if first_pre is None:first_pre=pre.copy()
            else:max_error=max(max_error,float(np.max(np.abs(pre-first_pre))))
            cue_length=len(tok(cue.cue_text,add_special_tokens=False).input_ids)
            for layer in MODEL_LAYERS:
                idx=layer-1
                clf,train_sd=probes[x][layer]
                pre_score=float(clf.decision_function(pre[idx].reshape(1,-1))[0])
                post_score=float(clf.decision_function(post[idx].reshape(1,-1))[0])
                u,v=pre[idx].astype(float),post[idx].astype(float)
                cosine=float(np.dot(u,v)/(max(np.linalg.norm(u)*np.linalg.norm(v),1e-12)))
                displacement=float(np.linalg.norm(v-u)/max(np.linalg.norm(u),1e-12))
                auc,is_valid=probe_map[(x,layer)]
                measurements.append(dict(
                    concept=x,replacement=cue.replacement,condition=condition,
                    cue_id=cue.cue_id,cue_text=cue.cue_text,
                    proposed_level=cue.proposed_level,measured_bin=cue.measured_bin,
                    semantic_distance=cue.semantic_distance,cue_token_count=cue_length,
                    prompt_tokens=n_tokens,pre_token_position=positions[0],
                    post_token_position=positions[1],layer=layer,
                    pre_probe_raw=pre_score,post_probe_raw=post_score,
                    pre_probe_std=pre_score/train_sd,post_probe_std=post_score/train_sd,
                    delta_probe_std=(post_score-pre_score)/train_sd,
                    hidden_cosine_distance=1.0-cosine,
                    hidden_relative_L2=displacement,
                    probe_auc=auc,validated=is_valid,prompt=p))
        precue_maxdiff[x+"_"+condition]=max_error
measure=pd.DataFrame(measurements)
assert len(measure)==len(CONCEPTS)*len(CONDITIONS)*12*24
assert not measure[["pre_probe_std","post_probe_std","delta_probe_std",
                    "hidden_cosine_distance","hidden_relative_L2"]].isna().any().any()
save_csv(measure,"06_layerwise_pre_post_measurements.csv")
print("Pre-cue invariance max |hidden state difference| by concept × condition:")
display(pd.DataFrame([{"key":k,"max_absolute_difference":v} for k,v in precue_maxdiff.items()]))
if max(precue_maxdiff.values())>0.01:
    warnings.warn("Unexpectedly different pre-cue hidden states. Check token alignment / numerical precision before interpretation.")
else:
    print("Passed pre-cue causal-prefix consistency check (<0.01 absolute difference).")
prompt_lengths=measure[measure.layer==MAIN_LAYER].groupby(
    ["condition","measured_bin"],as_index=False).agg(
    mean_total_tokens=("prompt_tokens","mean"),mean_cue_tokens=("cue_token_count","mean"),
    minimum=("prompt_tokens","min"),maximum=("prompt_tokens","max"))
save_csv(prompt_lengths,"07_token_length_diagnostics.csv")
display(prompt_lengths)

## 5. Paired contrasts — separating cue effects from suppression effects

Within each sequence define Δ = probe(post-cue) − probe(pre-cue), in units of the fitted probe's training decision-score SD. For the **same concept and same cue** define `instruction_interaction = Δsuppression − Δneutral`. Subtract the *same concept's* average interaction for its three **independently measured farthest cues** to form `DID_vs_unrelated`. A positive DID means a larger cue effect under suppression **relative to neutral and distant control cues**, not a higher absolute X score nor direct causal reactivation.

We keep per-cue values and concept-aggregated results. For layer-to-layer comparisons, the independent-probe validation pass set may vary, so figures must report n at each layer.

In [ ]:
idcols=["concept","replacement","cue_id","cue_text","proposed_level",
        "measured_bin","semantic_distance","cue_token_count","layer","validated","probe_auc"]
pivot=measure.pivot(index=idcols,columns="condition",
    values=["pre_probe_std","post_probe_std","delta_probe_std",
            "hidden_cosine_distance","hidden_relative_L2"]).reset_index()
pivot.columns=["_".join(str(a) for a in col if a!="") for col in pivot.columns]
assert len(pivot)==len(CONCEPTS)*12*24
pivot["interaction_delta"]=pivot.delta_probe_std_suppression-pivot.delta_probe_std_neutral
pivot["post_suppression_minus_neutral"]=pivot.post_probe_std_suppression-pivot.post_probe_std_neutral
pivot["pre_suppression_minus_neutral"]=pivot.pre_probe_std_suppression-pivot.pre_probe_std_neutral
unrelated=pivot[pivot.measured_bin=="unrelated"].groupby(
    ["concept","layer"],as_index=False).interaction_delta.mean().rename(
    columns={"interaction_delta":"unrelated_interaction_mean"})
pivot=pivot.merge(unrelated,on=["concept","layer"],validate="many_to_one")
pivot["DID_vs_unrelated"]=pivot.interaction_delta-pivot.unrelated_interaction_mean
assert pivot[pivot.measured_bin=="unrelated"].groupby(
    ["concept","layer"]).DID_vs_unrelated.mean().abs().max()<1e-7
save_csv(pivot,"08_matched_cue_layer_contrasts.csv")
concept_bins=pivot.groupby(["concept","layer","measured_bin","validated"],as_index=False).agg(
    mean_distance=("semantic_distance","mean"),
    mean_interaction=("interaction_delta","mean"),
    mean_DID=("DID_vs_unrelated","mean"),
    mean_neutral_delta=("delta_probe_std_neutral","mean"),
    mean_suppression_delta=("delta_probe_std_suppression","mean"),
    mean_neutral_post=("post_probe_std_neutral","mean"),
    mean_suppression_post=("post_probe_std_suppression","mean"),
    mean_hidden_distance_neutral=("hidden_cosine_distance_neutral","mean"),
    mean_hidden_distance_suppression=("hidden_cosine_distance_suppression","mean"),
    n_cues=("cue_id","count"))
save_csv(concept_bins,"09_concept_layer_distance_bin_summary.csv")
display(concept_bins[concept_bins.layer==MAIN_LAYER].pivot(
    index="concept",columns="measured_bin",values="mean_DID"
).reindex(columns=PROPOSED_LEVELS).round(3))

## 6. Layer-wise continuous-distance analysis with concept-level uncertainty

For **each validated concept/layer probe**, fit two exploratory within-concept relationships across the 12 independent cue texts: (a) simple slope of interaction Δ on continuous MiniLM distance; (b) sensitivity slope adjusting for Qwen cue token count. Then summarize slopes **across concepts**, resampling concepts rather than pretending 12 texts within one concept are independent samples. Some layers may have only 1–4 validated concepts; do **not** interpret their mean slope as a reliable layer effect.

The analysis is correlational. Our main prespecified layer is **24**, with all other layer comparisons exploratory and potentially affected by changing probe-validation membership.

In [ ]:
def bootstrap_concepts(values,seed=SEED,n_boot=N_BOOT):
    a=np.asarray(values,dtype=float);a=a[np.isfinite(a)]
    if len(a)<3:
        return (float(a.mean()) if len(a) else np.nan,np.nan,np.nan)
    rng=np.random.default_rng(seed)
    draws=rng.choice(a,size=(n_boot,len(a)),replace=True).mean(axis=1)
    return float(a.mean()),float(np.quantile(draws,.025)),float(np.quantile(draws,.975))

trends=[]
for (layer,x),g in pivot.groupby(["layer","concept"]):
    if len(g)!=12:raise ValueError(f"Expected 12 cues for {x}, layer {layer}")
    d=g.semantic_distance.to_numpy(float)
    y=g.interaction_delta.to_numpy(float)
    n=g.cue_token_count.to_numpy(float)
    dc=d-d.mean();yc=y-y.mean();nc=n-n.mean()
    simple=float(np.dot(dc,yc)/np.dot(dc,dc)) if np.dot(dc,dc)>1e-10 else np.nan
    controls=np.column_stack([dc,nc])
    beta=np.linalg.lstsq(controls,yc,rcond=None)[0]
    partial=float(beta[0]) if np.linalg.matrix_rank(controls)==2 else np.nan
    reliability=bool(g.validated.iloc[0])
    trends.append(dict(concept=x,layer=layer,validated=reliability,
        slope_distance_simple=simple,slope_distance_length_adjusted=partial,
        correlation_distance_length=float(np.corrcoef(d,n)[0,1]) if n.std()>0 else np.nan,
        n_cues=len(g)))
trends=pd.DataFrame(trends)
save_csv(trends,"10_distance_slope_by_concept_layer.csv")

slope_summary=[]
for layer in MODEL_LAYERS:
    for eligibility in ["all","validated"]:
        subset=trends[trends.layer==layer]
        if eligibility=="validated":subset=subset[subset.validated]
        for measure_name in ["slope_distance_simple","slope_distance_length_adjusted"]:
            m,lo,hi=bootstrap_concepts(subset[measure_name],seed=SEED+layer*17+
                (0 if eligibility=="all" else 1000)+(0 if measure_name=="slope_distance_simple" else 3))
            slope_summary.append(dict(layer=layer,analysis=eligibility,measure=measure_name,
                n_concepts=len(subset),estimate=m,ci95_low=lo,ci95_high=hi))
slope_summary=pd.DataFrame(slope_summary)
save_csv(slope_summary,"11_layerwise_distance_slopes_bootstrap.csv")
main=slope_summary[(slope_summary.layer==MAIN_LAYER)&
    (slope_summary.analysis=="validated")]
display(main.round(4))
print("At layer 24: direction of negative slope means smaller distance -> greater suppression-specific cue effect.")
print("The confidence interval is exploratory; the independent unit is concept (n≤8).")

## 7. Concept-level distance-bin DID: rank-based near/medium/far/unrelated

These distance bins are **defined using MiniLM quartiles per concept**, before any Qwen analysis. They are not independently verified semantic categories. The farthest bin becomes the unrelated-reference bin by construction; unlike the original hand-written category, it may contain a cue originally labeled near. Preserve both categorical descriptions in CSV. Use concept-cluster confidence intervals only when at least 3 eligible concepts exist.

In [ ]:
bin_rows=[]
for layer in MODEL_LAYERS:
    for bin_name in PROPOSED_LEVELS:
        for eligibility in ["all","validated"]:
            a=concept_bins[(concept_bins.layer==layer)&
                           (concept_bins.measured_bin==bin_name)]
            if eligibility=="validated":a=a[a.validated]
            estimate,lo,hi=bootstrap_concepts(a.mean_DID,
                seed=SEED+layer*19+PROPOSED_LEVELS.index(bin_name)*3+
                    (0 if eligibility=="all" else 900))
            bin_rows.append(dict(layer=layer,measured_bin=bin_name,analysis=eligibility,
                n_concepts=len(a),mean_DID=estimate,ci95_low=lo,ci95_high=hi))
bin_summary=pd.DataFrame(bin_rows)
save_csv(bin_summary,"12_distance_bin_layerwise_DID_bootstrap.csv")
display(bin_summary[(bin_summary.layer==MAIN_LAYER)&
    (bin_summary.analysis=="validated")].round(4))

# Key within-concept near-far contrast, and a separate all-concept diagnostic.
near_far=concept_bins.pivot(index=["concept","layer","validated"],
    columns="measured_bin",values="mean_DID").reset_index()
near_far["near_minus_far"]=near_far["near"]-near_far["far"]
save_csv(near_far,"13_near_vs_far_by_concept_layer.csv")
display(near_far[(near_far.layer==MAIN_LAYER)].loc[
    :,["concept","validated","near","far","near_minus_far"]].round(3))

## 8. Figure gallery — measured **24-layer** response profiles

Figures deliberately separate reliability, cue-induced state changes, suppression contrasts, semantic distance, and model-free representational displacement. A layer with fewer than 5 validated concepts is **not drawn as evidence of a semantic effect**. The plot legend explicitly reports concept n. The `all` records remain saved in CSV for transparent diagnosis.

In [ ]:
plt.rcParams.update({"font.size":10,"axes.spines.top":False,
                     "axes.spines.right":False,"savefig.facecolor":"white"})
palette={"near":"#164F73","medium":"#3485A4","far":"#81ADB8","unrelated":"#9A9A9A"}
# 01: validation AUC for all 8 concepts across all 24 layers
auc_grid=validation.pivot(index="concept",columns="layer",values="auc").reindex(
    index=CONCEPTS,columns=MODEL_LAYERS)
fig,ax=plt.subplots(figsize=(14,4.6))
image=ax.imshow(auc_grid.to_numpy(),cmap="Blues",vmin=0,vmax=1,aspect="auto")
ax.set(yticks=np.arange(len(CONCEPTS)),yticklabels=CONCEPTS,
       xticks=np.arange(24),xticklabels=MODEL_LAYERS,xlabel="Transformer layer",
       title="Held-out name-free X/Y probe ROC-AUC (all layers)")
for i in range(len(CONCEPTS)):
    for j in range(24):
        if auc_grid.iloc[i,j]>=AUC_CUTOFF:
            ax.text(j,i,"•",ha="center",va="center",color="white",fontsize=11)
fig.colorbar(image,ax=ax,label="ROC-AUC")
save_figure("01_all_layer_probe_validation_auc.png")

# 02: number of successfully validated concept probes per layer
fig,ax=plt.subplots(figsize=(10,3.4))
ax.bar(counts.layer,counts.n_passed,color="#246782",width=.75)
ax.axhline(MIN_VALIDATED_FOR_LAYER_PLOT,color="0.5",ls="--",
           label=f"Plot inclusion ≥{MIN_VALIDATED_FOR_LAYER_PLOT}/8")
ax.set(xlim=(.2,24.8),ylim=(0,8.7),xlabel="Transformer layer",
       ylabel="Passed concept probes (out of 8)",title="Interpretability coverage by layer")
ax.set_xticks(range(1,25,2));ax.legend()
save_figure("02_validated_concepts_by_layer.png")

# 03: condition-specific before → after probe score changes, by measured distance quartile
fig,axes=plt.subplots(1,2,figsize=(13,4.6),sharey=True)
for ax,cond in zip(axes,CONDITIONS):
    for cat in PROPOSED_LEVELS:
        group=measure[(measure.condition==cond)&(measure.measured_bin==cat)&(measure.validated)]
        # First average 3 cues within each concept, then 5+ concepts.
        a=group.groupby(["concept","layer"],as_index=False).delta_probe_std.mean()
        b=a.groupby("layer").agg(mean=("delta_probe_std","mean"),
                                 n=("concept","nunique")).reset_index()
        b=b[b.n>=MIN_VALIDATED_FOR_LAYER_PLOT]
        if len(b):
            ax.plot(b.layer,b["mean"],marker="o",markersize=3.0,lw=1.7,
                    color=palette[cat],label=cat)
    ax.axhline(0,color=".65",ls="--",lw=1)
    ax.set(xlim=(1,24),xlabel="Transformer layer",
           title=("Permissive reference" if cond=="neutral" else "Suppression"))
    ax.grid(alpha=.16)
axes[0].set_ylabel("Post − pre X/Y probe score (training SD units)")
if axes[1].get_legend_handles_labels()[1]:
    axes[1].legend(title="Measured quartile",loc="best")
fig.suptitle("Internal response to the cue at each layer (only validated probes; n≥5)")
save_figure("03_before_after_by_layer_and_instruction.png")

# 04: suppression-specific interaction, within concept, relative to farthest quartile
fig,ax=plt.subplots(figsize=(11,4.7))
for cat in PROPOSED_LEVELS:
    q=bin_summary[(bin_summary.analysis=="validated")&
                  (bin_summary.measured_bin==cat)&
                  (bin_summary.n_concepts>=MIN_VALIDATED_FOR_LAYER_PLOT)]
    if len(q):
        ax.plot(q.layer,q.mean_DID,marker="o",markersize=3,lw=1.9,
                color=palette[cat],label=cat)
        ax.fill_between(q.layer.to_numpy(float),q.ci95_low.to_numpy(float),
            q.ci95_high.to_numpy(float),color=palette[cat],alpha=.12)
ax.axhline(0,color=".55",ls="--")
ax.set(xlim=(1,24),xlabel="Transformer layer",ylabel="DID relative to distant cues (training SD units)",
       title="Suppression-specific cue response vs measured distance (concept bootstrap)")
ax.grid(alpha=.16)
if ax.get_legend_handles_labels()[1]:ax.legend(title="Measured quartile",loc="best")
save_figure("04_layerwise_suppression_distance_DID.png")

In [ ]:
# 05: continuous distance slopes; preserve negative and positive effects without clipping
fig,ax=plt.subplots(figsize=(10.2,4.6))
for measure_name,col,label in [
    ("slope_distance_simple","#205E7F","Unadjusted"),
    ("slope_distance_length_adjusted","#9B6F46","Adjusted for cue token count")]:
    q=slope_summary[(slope_summary.analysis=="validated")&
        (slope_summary.measure==measure_name)&
        (slope_summary.n_concepts>=MIN_VALIDATED_FOR_LAYER_PLOT)].copy()
    q=q.sort_values("layer")
    if len(q):
        ax.plot(q.layer,q.estimate,"-o",color=col,markersize=3,lw=1.8,label=label)
        ax.fill_between(q.layer.to_numpy(float),q.ci95_low.to_numpy(float),
            q.ci95_high.to_numpy(float),color=col,alpha=.13)
ax.axhline(0,color=".60",ls="--",lw=1)
ax.set(xlim=(1,24),xlabel="Transformer layer",
       ylabel="Concept-average slope of interaction vs semantic distance",
       title="Distance sensitivity across reliably decoded layers")
if ax.get_legend_handles_labels()[1]:ax.legend()
ax.grid(alpha=.18)
save_figure("05_continuous_distance_slopes_by_layer.png")

# 06: final-layer per-cue scatter; show original proposed group via marker edge only in CSV.
final=pivot[(pivot.layer==MAIN_LAYER)&pivot.validated]
fig,ax=plt.subplots(figsize=(9.0,5))
for cat in PROPOSED_LEVELS:
    q=final[final.measured_bin==cat]
    if len(q):ax.scatter(q.semantic_distance,q.DID_vs_unrelated,
        s=36,alpha=.82,color=palette[cat],label=cat)
ax.axhline(0,color=".6",ls="--")
ax.set(xlabel="Measured semantic distance (1 − MiniLM cosine)",
    ylabel="Suppression-specific DID vs distant cues",
    title=f"Layer {MAIN_LAYER}: per-cue distance / internal response (validated only)")
ax.grid(alpha=.16)
if ax.get_legend_handles_labels()[1]:ax.legend(title="Measured quartile")
save_figure("06_final_layer_distance_scatter.png")

# 07: per-concept profile: near minus far across all layers; mask poorly validated probes
nf=near_far.pivot(index="concept",columns="layer",values="near_minus_far").reindex(
    index=CONCEPTS,columns=MODEL_LAYERS)
passmask=validation.pivot(index="concept",columns="layer",values="valid_probe").reindex(
    index=CONCEPTS,columns=MODEL_LAYERS).fillna(False)
vals=np.where(passmask.to_numpy(dtype=bool),nf.to_numpy(float),np.nan)
finite=vals[np.isfinite(vals)]
bound=max(.1,float(np.quantile(np.abs(finite),.95))) if len(finite) else 1.0
fig,ax=plt.subplots(figsize=(14,4.8))
cmap=plt.get_cmap("coolwarm").copy();cmap.set_bad("#E8E8E8")
im=ax.imshow(np.ma.masked_invalid(vals),aspect="auto",cmap=cmap,vmin=-bound,vmax=bound)
ax.set(yticks=np.arange(len(CONCEPTS)),yticklabels=CONCEPTS,
       xticks=np.arange(24),xticklabels=MODEL_LAYERS,xlabel="Transformer layer",
       title="Per-concept near-minus-far suppression contrast (gray = probe AUC<0.75)")
fig.colorbar(im,ax=ax,label="Near − far DID (training SD units)")
save_figure("07_concept_by_layer_near_minus_far.png")

# 08: probe-free hidden-state change. Reflects any contextual/positional change, not X-specific meaning.
fig,axes=plt.subplots(1,2,figsize=(12,4.6),sharey=True)
for ax,cond in zip(axes,CONDITIONS):
    for cat in PROPOSED_LEVELS:
        g=measure[(measure.condition==cond)&(measure.measured_bin==cat)]
        means=g.groupby(["concept","layer"],as_index=False).hidden_cosine_distance.mean()
        avg=means.groupby("layer").hidden_cosine_distance.mean()
        ax.plot(avg.index,avg.values,color=palette[cat],lw=1.9,label=cat)
    ax.set(xlabel="Transformer layer",title=cond.capitalize())
    ax.grid(alpha=.15)
axes[0].set_ylabel("1 − cosine(hidden pre, hidden post) (not concept-specific)")
axes[1].legend(title="Measured quartile",loc="best")
fig.suptitle("Probe-free representational displacement across all 24 layers")
save_figure("08_probe_free_hidden_displacement.png")
print("Saved 8 figures to",OUT)

## 9. Integrity checks and recorded interpretation limits

This cell fails loudly if any raw data or controlled comparison is missing; it records the number of reliable probes at every layer, model package versions, and immutable input-bank references. Before interpreting any result, check that the *same-prompt pre-cue state* is invariant to cue changes, and whether measured semantic-distance ranks disagree with the proposed level names.

**Strict interpretation:** AUC is classification discrimination, not calibrated probability or proof of activation; similar late-layer scores can coexist with output suppression. Hidden displacement measures general contextual change and need not reflect the forbidden concept. Any correlation is observational, not a causal estimate.

In [ ]:
assert len(cues)==8*12
assert cues.groupby(["concept","measured_bin"]).size().eq(3).all()
assert len(validation)==8*24
assert len(measure)==8*2*12*24
assert len(pivot)==8*12*24
assert len(trends)==8*24
assert set(MODEL_LAYERS)==set(range(1,25))
assert all(np.isfinite(measure[k]).all() for k in [
    "pre_probe_std","post_probe_std","delta_probe_std",
    "hidden_cosine_distance","hidden_relative_L2"])
assert pivot[pivot.measured_bin=="unrelated"].groupby(
    ["concept","layer"]).DID_vs_unrelated.mean().abs().max()<1e-7
summary_metadata={
    "title":"1019 v2 — within-sequence cue response and layer-wise suppressed concept",
    "run_id":RUN_ID,"created_utc":datetime.now(timezone.utc).isoformat(),
    "llm":MODEL_ID,"semantic_encoder":EMBED_ID,
    "seed":SEED,"layers":MODEL_LAYERS,"focus_layer":MAIN_LAYER,
    "concepts":CONCEPTS,"n_cues_per_concept":12,
    "n_instruction_conditions":len(CONDITIONS),"conditions":CONDITIONS,
    "auc_cutoff":AUC_CUTOFF,"minimum_validated_for_layer_plot":MIN_VALIDATED_FOR_LAYER_PLOT,
    "validated_probe_count_by_layer":{str(int(r.layer)):int(r.n_passed) for r in counts.itertuples()},
    "n_probes_passed_main_layer":int(counts.loc[counts.layer==MAIN_LAYER,"n_passed"].iloc[0]),
    "proposed_distance_order_confirmed_in":int(audit_wide.proposed_order_confirmed.sum()),
    "distance_metric":"1 - cosine(two target sentences centroid, cue sentence), MiniLM mean pooling",
    "distance_bins":"within-concept ranked quartiles; 3 cues per bin; independent of Qwen and probes",
    "readout_token":"last token covering STATE: anchor using fast-tokenizer character offsets",
    "measurement_type":"single causal forward pass, pre-cue vs post-cue markers, no generation",
    "pre_cue_invariance_maxdiff":precue_maxdiff,
    "temperature_or_random_sampling":None,
    "bootstrap_unit":"target concept (8 pairs)",
    "n_bootstrap":N_BOOT,
    "stimulus_urls":{"cues":STIMULI_URL,"probe_description":DESCRIPTION_URL},
    "input_sha256":INPUT_HASHES,
    "runtime":{"device":DEVICE,"python":sys.version,"torch":torch.__version__,
        "transformers":__import__("transformers").__version__,
        "pandas":pd.__version__,"numpy":np.__version__},
    "limitations":[
        "X/Y probe measures relative discriminability, not absolute concept activation.",
        "Suppression and permissive-reference instructions mention X once each but are not textually identical.",
        "Pre/post are token positions in one causal input, not elapsed thinking time.",
        "Embedding-based distance is not validated by independent human ratings.",
        "Proposed stimulus levels may not match empirical rank bins; retain both labels.",
        "Only eight hand-selected concept pairs and twelve cue texts per pair.",
        "Name-free probe descriptions share semantics with some cues; stronger independent validation is needed.",
        "Layer means use different concepts when eligibility filters differ across layers.",
        "Probe-free displacement measures general hidden change, not the suppressed concept.",
        "Cues vary in token length; adjusted slopes are sensitivity checks, not perfect length matching.",
        "Intervals are exploratory concept bootstraps, not preregistered confirmatory inference.",
        "No causal intervention, activation steering, or external-model replication."
    ]
}
(OUT/"metadata.json").write_text(json.dumps(summary_metadata,indent=2,
                                           ensure_ascii=False),encoding="utf-8")
readme="\n".join([
 "# 1019 v2 — Layer-wise semantic distance vs concept suppression",
 "",f"Run: {RUN_ID}",f"LLM: {MODEL_ID}",f"Encoder: {EMBED_ID}","",
 "## Main analysis",
 "X vs Y probes at 24 layers; same-prompt pre- and post-cue STATE: markers.",
 "For each cue: interaction = (after-before)_suppression - (after-before)_neutral.",
 "DID_vs_unrelated subtracts each concept's three farthest measured cues.",
 "Plots of semantic X/Y readouts require held-out description AUC≥0.75 and at least 5 valid concept probes.",
 "Semantic bin labels are quartiles from an independent sentence encoder, not verified human ratings.","",
 "## CSV and PNG outputs",
 "01_stimulus_distances_and_ranks.csv: independent continuous distances and both cue labels",
 "02_proposed_distance_audit.csv: which original distance orders held",
 "03_probe_validation_all_layers.csv: held-out AUC and accuracy",
 "04_probe_validation_flags.csv / 05_probe_validation_per_layer.csv",
 "06_layerwise_pre_post_measurements.csv: raw and standardized X/Y scores plus vector displacement",
 "07_token_length_diagnostics.csv: prompt length differences",
 "08_matched_cue_layer_contrasts.csv: delta, instruction interaction, DID",
 "09_concept_layer_distance_bin_summary.csv: concept-level near/medium/far/unrelated",
 "10_distance_slope_by_concept_layer.csv: concept-specific distance sensitivity",
 "11_layerwise_distance_slopes_bootstrap.csv: concept bootstrap across layers",
 "12_distance_bin_layerwise_DID_bootstrap.csv: measured-bin contrast intervals",
 "13_near_vs_far_by_concept_layer.csv: validated mask and near-minus-far profile",
 "01..08_*.png: 8 diagnostic and scientific figures",
 "stimuli_v2.csv and probe_description_bank_v2.json: exact input copies",
 "metadata.json: experiment and limitations","",
 "## Interpretation",
 "Positive DID: suppression differs from neutral in cue-driven change relative to distant cues.",
 "It does NOT mean that the suppressed concept is absolutely stronger than normal.",
 "Do not claim causal reactivation, knowledge erasure, or reliable early-layer trends without probe validity.",""
])+"\n"
(OUT/"README.md").write_text(readme,encoding="utf-8")
zip_path=OUT.parent/(RUN_ID+"_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,arcname=f"1019/results_v2/{RUN_ID}/{p.name}")
print("Experiment completed; CSV:",len(list(OUT.glob("*.csv"))),
      "PNG:",len(list(OUT.glob("*.png"))),"JSON:",len(list(OUT.glob("*.json"))))
print("Archive:",zip_path,"size:",round(zip_path.stat().st_size/1024,1),"KiB")

## 10. Secure automatic GitHub upload into `1019/results_v2/<UTC-run-id>/`

Create a Colab **Secret named `GITHUB_RESULTS_TOKEN`**, grant notebook access, and ensure the token has repository Contents: **Read and write** permission for `ryouy/GraduationThesis`. The notebook does **not** store tokens in metadata, CSV, PNG, or GitHub source. GitHub uploads are optional: a missing secret does not stop saving local results. Re-running this cell uses a GitHub blob SHA to update rather than failing on an already-uploaded path.

In [ ]:
def get_token():
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_RESULTS_TOKEN")
    except Exception:
        return os.environ.get("GITHUB_RESULTS_TOKEN")
if UPLOAD_TO_GITHUB:
    token=get_token()
    if not token:
        print("GitHub upload skipped: GITHUB_RESULTS_TOKEN not available in Colab Secrets.")
        print("Local results and ZIP remain at:",OUT)
    else:
        headers={"Authorization":f"Bearer {token}",
                 "Accept":"application/vnd.github+json",
                 "X-GitHub-Api-Version":"2022-11-28"}
        saved,failed=[],[]
        candidates=[p for p in sorted(OUT.iterdir()) if p.is_file() and
                    p.suffix.lower() in {".csv",".png",".json",".md"}]
        for file in tqdm(candidates,desc="Upload v2 CSV/PNG/metadata"):
            remote=f"1019/results_v2/{RUN_ID}/{file.name}"
            url=f"https://api.github.com/repos/{REPO}/contents/{remote}"
            payload={"message":f"research: save 1019 v2 run {RUN_ID} ({file.name})",
                     "content":base64.b64encode(file.read_bytes()).decode("ascii"),"branch":BRANCH}
            try:
                existing=requests.get(url,headers=headers,timeout=30)
                if existing.status_code==200:
                    payload["sha"]=existing.json()["sha"]
                elif existing.status_code!=404:
                    raise RuntimeError(f"Pre-upload lookup HTTP {existing.status_code}")
                response=requests.put(url,headers=headers,json=payload,timeout=90)
                if response.status_code not in (200,201):
                    raise RuntimeError(f"Upload HTTP {response.status_code}: "
                        "check token Contents write scope and repository branch")
                saved.append(remote)
            except Exception as e:
                failed.append((remote,str(e)));print("WARNING:",remote,str(e))
        print(f"Uploaded {len(saved)}/{len(candidates)} v2 result files.")
        if saved:print(f"https://github.com/{REPO}/tree/{BRANCH}/1019/results_v2/{RUN_ID}")
        if failed:print("Incomplete upload; use the local ZIP or rerun the upload cell.")
        del token,headers
else:
    print("UPLOAD_TO_GITHUB=False: outputs remain in",OUT)

In [ ]:
# Optional manual download of all CSV/PNG/JSON/README files as one archive.
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print("Not in Colab. Your result ZIP:",zip_path)